Connected to nb-cuda (Python 3.13.11)

In [ ]:
####################################

import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import sys, subprocess, importlib, warnings, math, os

from pathlib import Path

import torch
import scvi

In [ ]:
##
# pip3 install -U scvi-tools[cuda]  # gets jax and jaxlib, updates cuda
# pip3 install -U scib-metrics
#
#
####################################

###########
# ## STEP 0. Workspace Setup

# set general folder paths
HOME = Path.home()
WS_ROOT = HOME / "workspace"
DATA_DIR = WS_ROOT / "data"
WS_FILES = WS_ROOT / "ws_files"

In [ ]:
## Build and set path to desired dataset
DATASETS_PATH = WS_ROOT / "01_PMDBS" / "PMDBS_sc_rnaseq"

workflow = "pmdbs_sc_rnaseq"
dataset_team = "cohort"
dataset_source = "pmdbs"
dataset_type = "sc-rnaseq"

bucket_name = f"asap-curated-{dataset_team}-{dataset_source}-{dataset_type}"
dataset_name = f"asap-{dataset_team}-{dataset_source}-{dataset_type}"

dataset_path = DATASETS_PATH / bucket_name / workflow
print("Dataset Path:", dataset_path)

# Build the folder path to the cohort analysis directory
cohort_analysis_path = dataset_path / "cohort_analysis"

# Preview the directory contents
# Define a local path for workshop files
local_data_path = WS_FILES / "case_study_01"

# map my cells directories
mapmycells_input_dir = local_data_path / "mapmycells/input"
mapmycells_output_dir = local_data_path / "mapmycells/output"

# other directories
resources_path = local_data_path / "resources"
plots_path = local_data_path / "output_plots"
output_path = local_data_path / "output_matrices"

# Make sure the directories exists
os.makedirs(mapmycells_input_dir, exist_ok=True)
os.makedirs(mapmycells_output_dir, exist_ok=True)
os.makedirs(resources_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)
os.makedirs(plots_path, exist_ok=True)

# Create the directory if it doesn't already exist
if not local_data_path.exists():
    local_data_path.mkdir(parents=True)

print(f"Local data directory ready at: {local_data_path}")

Dataset Path: /home/ergonyc/workspace/01_PMDBS/PMDBS_sc_rnaseq/asap-curated-cohort-pmdbs-sc-rnaseq/pmdbs_sc_rnaseq
Local data directory ready at: /home/ergonyc/workspace/ws_files/case_study_01


In [ ]:
###########

In [ ]:
sn_full_raw_filename = local_data_path / f"asap-{dataset_team}.SN.01_full_raw.h5ad"

In [ ]:
sn_full_norm_filename = local_data_path / f"asap-{dataset_team}.SN.02_raw_norm.h5ad"

In [ ]:
#################################
sn_processed_filename = local_data_path / f"asap-{dataset_team}.SN.02_processed.h5ad"

# Save the anndata object
sn_integrated_filename = local_data_path / f"asap-{dataset_team}.SN.03_scvi.h5ad"

# output file neame
sn_mmc_pheno_filename = (
    local_data_path / f"asap-{dataset_team}.SN.04_mmc_processed.h5ad"
)


# have to use ENSG ids for this mapmycells taxonomy
# prep data

In [ ]:
###########
adata = sc.read_h5ad(sn_mmc_pheno_filename)
# ## STEP 3. make SN integrate with scVI (.SN.03_scvi.h5ad)
batch_key = "sample"
n_layers = 2
n_latent = n_comps  # defined above
###
print(torch.cuda.is_available())

scvi.settings.seed = 0
torch.set_float32_matmul_precision("high")


scvi_model_filename = local_data_path / f"asap-{dataset_team}.SN.03_scvi_model.pkl"
vae = scvi.model.SCVI.load(scvi_model_filename)

NameError: name 'n_comps' is not defined

In [ ]:
def label_with_scanvi(
    adata: ad.AnnData, model: scvi.model.SCVI, num_workers: int, workflow_name: str
) -> tuple[ad.AnnData, scvi.model.SCANVI]:
    """
    Fit scANVI model to AnnData object
    """

    # Fixed parameters
    scanvi_epochs = 300
    batch_size = 1024
    accelerator = "gpu"
    dispersion = "gene-cell"  # "gene"
    gene_likelihood = "zinb"
    latent_distribution = "normal"
    early_stopping = True
    early_stopping_patience = 20

    # if adata.n_obs > threshold_cells:
    #     plan_kwargs = {"lr": 1e-4}
    #     gradient_clip_val = 5.0
    #     print(f"AnnData object contains {adata.n_obs} which is > {threshold_cells}")
    #     print(f"--- Using learning rate: {plan_kwargs}")
    #     print(f"--- Using gradient clipping: {gradient_clip_val}")
    # else:
    # Defaults
    plan_kwargs = {"lr": 1e-3}
    gradient_clip_val = None
    # print(f"AnnData object contains {adata.n_obs} which is < {threshold_cells}")
    # print(f"--- Using default learning rate: {plan_kwargs}")
    # print(f"--- Using default gradient clipping: {gradient_clip_val}")

    print("Generating scANVI model from scVI")
    scanvi_model = scvi.model.SCANVI.from_scvi_model(
        model,
        adata=adata,
        labels_key="cell_type",
        unlabeled_category="Unknown",
    )

    print("Training scANVI model")
    scanvi_model.train(
        accelerator=accelerator,
        max_epochs=scanvi_epochs,
        early_stopping=early_stopping,
        early_stopping_patience=early_stopping_patience,
        datasplitter_kwargs={"num_workers": num_workers},
        gradient_clip_val=gradient_clip_val,
        plan_kwargs=plan_kwargs,
    )

    print("Generating scANVI latents and predictions")
    adata.obsm[args.latent_key] = scanvi_model.get_latent_representation(adata)
    adata.obs[args.predictions_key] = scanvi_model.predict(adata)

    return (adata, scanvi_model)


num_workers = 0  # Pytorch bug unable to mmap solution https://github.com/pytorch/pytorch/issues/92134
scvi.settings.dl_num_workers = num_workers
print(f"Using {scvi.settings.dl_num_workers} workers")

# 4. Get scANVI model
workflow_name = "case_study_01"
adata, scanvi_model = label_with_scanvi(adata, vae, num_workers, workflow_name)
# 5. Save the integrated adata and scANVI model


scanvi_model_filename = local_data_path / f"asap-{dataset_team}.SN.05_scanvi_model"

scanvi_model.save(scanvi_model_filename, overwrite=True)
# 6. Save the latent space
# output file neame
sn_scanvi_filename = local_data_path / f"asap-{dataset_team}.SN.05_scanvi.h5ad"

adata.write_h5ad(filename=args.adata_output, compression="gzip")
# 7. Save the cell types to feather
# adata.obs[[args.predictions_key]].to_feather(args.output_cell_types_file, compression="gzip")
# 7. Save the cell types to parquet

output_cell_types_file = (
    local_data_path / f"asap-{dataset_team}.SN.05_scanvi_cell_types.parquet"
)

adata.obs[[args.predictions_key]].to_parquet(output_cell_types_file, compression="gzip")

: 

In [ ]:
n_comps = 30
n_latent = n_comps  # defined above
###
print(torch.cuda.is_available())

scvi.settings.seed = 0
torch.set_float32_matmul_precision("high")


scvi_model_filename = local_data_path / f"asap-{dataset_team}.SN.03_scvi_model.pkl"
vae = scvi.model.SCVI.load(scvi_model_filename)

Seed set to 0


True
INFO     No backup URL provided for missing file                                                                   
         /home/ergonyc/workspace/ws_files/case_study_01/asap-cohort.SN.03_scvi_model.pkl/model.pt                  


ValueError: Failed to load model file at /home/ergonyc/workspace/ws_files/case_study_01/asap-cohort.SN.03_scvi_model.pkl/model.pt. If attempting to load a saved model from <v0.15.0, please use the util function `convert_legacy_save` to convert to an updated format.

In [ ]:
scvi_model_filename = local_data_path / f"asap-{dataset_team}.SN.03_scvi_model"
vae = scvi.model.SCVI.load(scvi_model_filename)

INFO     File /home/ergonyc/workspace/ws_files/case_study_01/asap-cohort.SN.03_scvi_model/model.pt already         
         downloaded                                                                                                


/home/ergonyc/mambaforge/envs/nb-cuda/lib/python3.13/site-packages/scvi/model/base/_base_model.py:869: UserWarning: Save path contains no saved anndata and no adata was passed. Model will be loaded without anndata.
  ) = _load_saved_files(


In [ ]:
adata

AnnData object with n_obs × n_vars = 258505 × 2764
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'n_genes_by_counts', 'total_counts', 'total_counts_rb', 'pct_counts_rb', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'sample', 'batch', 'team', 'dataset', 'batch_id', 'S_score', 'G2M_score', 'phase', 'brain_region', 'brain_region_simple', 'case_id', 'condition_id', 'region_level_1', 'region_level_2', '_cell_type', '_phenotype', '_rho', '_prob', '_class_name', '_subclass_name', '_supertype_name', '__scvi_batch', '__scvi_labels', '_C_scANVI', '_leiden_res_0.05', '_leiden_res_0.10', '_leiden_res_0.20', '_leiden_res_0.40', '_scvi_batch', '_scvi_labels', 'atlas_identifier', 'leiden_2', 'leiden', 'leiden_05', 'cell_type', 'phenotype', 'rho', 'prob', 'class_name', 'subclass_name', 'supertype_name'
    var: 'feature_type', 'genome', 'gene_id', 'mt', 'rb', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'n_cells'
    uns: '_scvi_manage

In [ ]:
adata.obsm["X_scvi"]

array([[-2.2359673e-02, -2.9435724e-02, -8.0053210e-02, ...,
        -2.4471488e-02,  3.3916697e-02, -2.2506591e-02],
       [-1.3455398e-02, -5.0719166e-03,  1.9044524e-01, ...,
        -1.8456716e-02, -2.4266597e-03, -1.1148471e-02],
       [-9.8067708e-03, -7.7295033e-03, -2.8087050e-01, ...,
        -8.6325416e-03,  5.6680124e-03, -1.6347856e-03],
       ...,
       [-2.8191013e-03,  4.3512648e-04,  1.5644103e-01, ...,
        -2.4350672e-03, -7.3622270e-03,  3.2036551e-03],
       [-1.0208799e-02,  7.3334533e-03,  5.3234643e-01, ...,
         3.4662788e-03, -2.1518571e-02,  5.4869191e-03],
       [-1.7047584e-02, -5.7002008e-03, -2.4066907e-01, ...,
        -2.1064677e-03, -1.2775771e-03,  2.3079286e-03]],
      shape=(258505, 30), dtype=float32)

In [ ]:
n_comps = adata.obsm["X_pca"].shape[1]
n_latent = n_comps  # defined above

In [ ]:
def label_with_scanvi(
    adata: ad.AnnData, model: scvi.model.SCVI, num_workers: int, workflow_name: str
) -> tuple[ad.AnnData, scvi.model.SCANVI]:
    """
    Fit scANVI model to AnnData object
    """

    # Fixed parameters
    scanvi_epochs = 300
    batch_size = 1024
    accelerator = "gpu"
    dispersion = "gene-cell"  # "gene"
    gene_likelihood = "zinb"
    latent_distribution = "normal"
    early_stopping = True
    early_stopping_patience = 20

    # if adata.n_obs > threshold_cells:
    #     plan_kwargs = {"lr": 1e-4}
    #     gradient_clip_val = 5.0
    #     print(f"AnnData object contains {adata.n_obs} which is > {threshold_cells}")
    #     print(f"--- Using learning rate: {plan_kwargs}")
    #     print(f"--- Using gradient clipping: {gradient_clip_val}")
    # else:
    # Defaults
    plan_kwargs = {"lr": 1e-3}
    gradient_clip_val = None
    # print(f"AnnData object contains {adata.n_obs} which is < {threshold_cells}")
    # print(f"--- Using default learning rate: {plan_kwargs}")
    # print(f"--- Using default gradient clipping: {gradient_clip_val}")

    print("Generating scANVI model from scVI")
    scanvi_model = scvi.model.SCANVI.from_scvi_model(
        model,
        adata=adata,
        labels_key="cell_type",
        unlabeled_category="Unknown",
    )

    print("Training scANVI model")
    scanvi_model.train(
        accelerator=accelerator,
        max_epochs=scanvi_epochs,
        early_stopping=early_stopping,
        early_stopping_patience=early_stopping_patience,
        datasplitter_kwargs={"num_workers": num_workers},
        gradient_clip_val=gradient_clip_val,
        plan_kwargs=plan_kwargs,
    )

    print("Generating scANVI latents and predictions")
    adata.obsm[args.latent_key] = scanvi_model.get_latent_representation(adata)
    adata.obs[args.predictions_key] = scanvi_model.predict(adata)

    return (adata, scanvi_model)

NameError: name 'ad' is not defined

In [ ]:
def label_with_scanvi(
    adata: sc.AnnData, model: scvi.model.SCVI, num_workers: int, workflow_name: str
) -> tuple[sc.AnnData, scvi.model.SCANVI]:
    """
    Fit scANVI model to AnnData object
    """

    # Fixed parameters
    scanvi_epochs = 300
    batch_size = 1024
    accelerator = "gpu"
    dispersion = "gene-cell"  # "gene"
    gene_likelihood = "zinb"
    latent_distribution = "normal"
    early_stopping = True
    early_stopping_patience = 20

    # if adata.n_obs > threshold_cells:
    #     plan_kwargs = {"lr": 1e-4}
    #     gradient_clip_val = 5.0
    #     print(f"AnnData object contains {adata.n_obs} which is > {threshold_cells}")
    #     print(f"--- Using learning rate: {plan_kwargs}")
    #     print(f"--- Using gradient clipping: {gradient_clip_val}")
    # else:
    # Defaults
    plan_kwargs = {"lr": 1e-3}
    gradient_clip_val = None
    # print(f"AnnData object contains {adata.n_obs} which is < {threshold_cells}")
    # print(f"--- Using default learning rate: {plan_kwargs}")
    # print(f"--- Using default gradient clipping: {gradient_clip_val}")

    print("Generating scANVI model from scVI")
    scanvi_model = scvi.model.SCANVI.from_scvi_model(
        model,
        adata=adata,
        labels_key="cell_type",
        unlabeled_category="Unknown",
    )

    print("Training scANVI model")
    scanvi_model.train(
        accelerator=accelerator,
        max_epochs=scanvi_epochs,
        early_stopping=early_stopping,
        early_stopping_patience=early_stopping_patience,
        datasplitter_kwargs={"num_workers": num_workers},
        gradient_clip_val=gradient_clip_val,
        plan_kwargs=plan_kwargs,
    )

    print("Generating scANVI latents and predictions")
    adata.obsm[args.latent_key] = scanvi_model.get_latent_representation(adata)
    adata.obs[args.predictions_key] = scanvi_model.predict(adata)

    return (adata, scanvi_model)

In [ ]:
num_workers = 0  # Pytorch bug unable to mmap solution https://github.com/pytorch/pytorch/issues/92134
scvi.settings.dl_num_workers = num_workers
print(f"Using {scvi.settings.dl_num_workers} workers")

# 4. Get scANVI model
workflow_name = "case_study_01"
adata, scanvi_model = label_with_scanvi(adata, vae, num_workers, workflow_name)
# 5. Save the integrated adata and scANVI model

Using 0 workers
Generating scANVI model from scVI
INFO     Model was loaded without AnnData. Setting up provided AnnData using saved registry.                       
Training scANVI model
INFO     Training for 300 epochs.                                                                                  


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/ergonyc/mambaforge/envs/nb-cuda/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/ergonyc/mambaforge/envs/nb-cuda/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training:   0%|          | 0/300 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=300` reached.


Generating scANVI latents and predictions


NameError: name 'args' is not defined

In [ ]:
scanvi_model_filename = local_data_path / f"asap-{dataset_team}.SN.05_scanvi_model"

scanvi_model.save(scanvi_model_filename, overwrite=True)
# 6. Save the latent space
# output file neame
sn_scanvi_filename = local_data_path / f"asap-{dataset_team}.SN.05_scanvi.h5ad"

adata.write_h5ad(filename=args.adata_output, compression="gzip")
# 7. Save the cell types to feather
# adata.obs[[args.predictions_key]].to_feather(args.output_cell_types_file, compression="gzip")
# 7. Save the cell types to parquet

output_cell_types_file = (
    local_data_path / f"asap-{dataset_team}.SN.05_scanvi_cell_types.parquet"
)

adata.obs[[args.predictions_key]].to_parquet(output_cell_types_file, compression="gzip")

: 

Connected to nb-cuda (Python 3.13.11)

In [ ]:
####################################

import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import sys, subprocess, importlib, warnings, math, os

from pathlib import Path

import torch
import scvi

In [ ]:
##
# pip3 install -U scvi-tools[cuda]  # gets jax and jaxlib, updates cuda
# pip3 install -U scib-metrics
#
#
####################################

###########
# ## STEP 0. Workspace Setup

# set general folder paths
HOME = Path.home()
WS_ROOT = HOME / "workspace"
DATA_DIR = WS_ROOT / "data"
WS_FILES = WS_ROOT / "ws_files"

In [ ]:
## Build and set path to desired dataset
DATASETS_PATH = WS_ROOT / "01_PMDBS" / "PMDBS_sc_rnaseq"

workflow = "pmdbs_sc_rnaseq"
dataset_team = "cohort"
dataset_source = "pmdbs"
dataset_type = "sc-rnaseq"

bucket_name = f"asap-curated-{dataset_team}-{dataset_source}-{dataset_type}"
dataset_name = f"asap-{dataset_team}-{dataset_source}-{dataset_type}"

dataset_path = DATASETS_PATH / bucket_name / workflow
print("Dataset Path:", dataset_path)

# Build the folder path to the cohort analysis directory
cohort_analysis_path = dataset_path / "cohort_analysis"

# Preview the directory contents
# Define a local path for workshop files
local_data_path = WS_FILES / "case_study_01"

# map my cells directories
mapmycells_input_dir = local_data_path / "mapmycells/input"
mapmycells_output_dir = local_data_path / "mapmycells/output"

# other directories
resources_path = local_data_path / "resources"
plots_path = local_data_path / "output_plots"
output_path = local_data_path / "output_matrices"

# Make sure the directories exists
os.makedirs(mapmycells_input_dir, exist_ok=True)
os.makedirs(mapmycells_output_dir, exist_ok=True)
os.makedirs(resources_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)
os.makedirs(plots_path, exist_ok=True)

# Create the directory if it doesn't already exist
if not local_data_path.exists():
    local_data_path.mkdir(parents=True)

print(f"Local data directory ready at: {local_data_path}")

Dataset Path: /home/ergonyc/workspace/01_PMDBS/PMDBS_sc_rnaseq/asap-curated-cohort-pmdbs-sc-rnaseq/pmdbs_sc_rnaseq
Local data directory ready at: /home/ergonyc/workspace/ws_files/case_study_01


In [ ]:
###########

In [ ]:
sn_full_raw_filename = local_data_path / f"asap-{dataset_team}.SN.01_full_raw.h5ad"

In [ ]:
sn_full_norm_filename = local_data_path / f"asap-{dataset_team}.SN.02_raw_norm.h5ad"

In [ ]:
#################################
sn_processed_filename = local_data_path / f"asap-{dataset_team}.SN.02_processed.h5ad"

# Save the anndata object
sn_integrated_filename = local_data_path / f"asap-{dataset_team}.SN.03_scvi.h5ad"

# output file neame
sn_mmc_pheno_filename = (
    local_data_path / f"asap-{dataset_team}.SN.04_mmc_processed.h5ad"
)


# have to use ENSG ids for this mapmycells taxonomy
# prep data

In [ ]:
###########
adata = sc.read_h5ad(sn_mmc_pheno_filename)
# ## STEP 3. make SN integrate with scVI (.SN.03_scvi.h5ad)
batch_key = "sample"
n_layers = 2
n_comps = adata.obsm["X_pca"].shape[1]
n_latent = n_comps  # defined above
###
print(torch.cuda.is_available())

scvi.settings.seed = 0
torch.set_float32_matmul_precision("high")


scvi_model_filename = local_data_path / f"asap-{dataset_team}.SN.03_scvi_model"
vae = scvi.model.SCVI.load(scvi_model_filename)

Seed set to 0


True
INFO     File /home/ergonyc/workspace/ws_files/case_study_01/asap-cohort.SN.03_scvi_model/model.pt already         
         downloaded                                                                                                


/home/ergonyc/mambaforge/envs/nb-cuda/lib/python3.13/site-packages/scvi/model/base/_base_model.py:869: UserWarning: Save path contains no saved anndata and no adata was passed. Model will be loaded without anndata.
  ) = _load_saved_files(


In [ ]:
def label_with_scanvi(
    adata: sc.AnnData,
    model: scvi.model.SCVI,
    num_workers: int,
    latent_key: str | None = None,
    predictions_key: str | None = None,
    workflow_name: str = "generic_000",
) -> tuple[sc.AnnData, scvi.model.SCANVI]:
    """
    Fit scANVI model to AnnData object
    """

    # Fixed parameters
    scanvi_epochs = 300
    batch_size = 1024
    accelerator = "gpu"
    dispersion = "gene-cell"  # "gene"
    gene_likelihood = "zinb"
    latent_distribution = "normal"
    early_stopping = True
    early_stopping_patience = 20

    if latent_key is None:
        latent_key = f"X_scanvi"
    if predictions_key is None:
        predictions_key = f"cell_type"

    # if adata.n_obs > threshold_cells:
    #     plan_kwargs = {"lr": 1e-4}
    #     gradient_clip_val = 5.0
    #     print(f"AnnData object contains {adata.n_obs} which is > {threshold_cells}")
    #     print(f"--- Using learning rate: {plan_kwargs}")
    #     print(f"--- Using gradient clipping: {gradient_clip_val}")
    # else:
    # Defaults
    plan_kwargs = {"lr": 1e-3}
    gradient_clip_val = None
    # print(f"AnnData object contains {adata.n_obs} which is < {threshold_cells}")
    # print(f"--- Using default learning rate: {plan_kwargs}")
    # print(f"--- Using default gradient clipping: {gradient_clip_val}")

    print("Generating scANVI model from scVI")
    scanvi_model = scvi.model.SCANVI.from_scvi_model(
        model,
        adata=adata,
        labels_key="cell_type",
        unlabeled_category="Unknown",
    )

    print("Training scANVI model")
    scanvi_model.train(
        accelerator=accelerator,
        max_epochs=scanvi_epochs,
        early_stopping=early_stopping,
        early_stopping_patience=early_stopping_patience,
        datasplitter_kwargs={"num_workers": num_workers},
        gradient_clip_val=gradient_clip_val,
        plan_kwargs=plan_kwargs,
    )

    print("Generating scANVI latents and predictions")
    adata.obsm[latent_key] = scanvi_model.get_latent_representation(adata)
    adata.obs[predictions_key] = scanvi_model.predict(adata)

    return (adata, scanvi_model)

In [ ]:
num_workers = 0  # Pytorch bug unable to mmap solution https://github.com/pytorch/pytorch/issues/92134
scvi.settings.dl_num_workers = num_workers
print(f"Using {scvi.settings.dl_num_workers} workers")

# 4. Get scANVI model
workflow_name = "case_study_01"
adata, scanvi_model = label_with_scanvi(adata, vae, num_workers, workflow_name)
# 5. Save the integrated adata and scANVI model

Using 0 workers
Generating scANVI model from scVI
INFO     Model was loaded without AnnData. Setting up provided AnnData using saved registry.                       
Training scANVI model
INFO     Training for 300 epochs.                                                                                  


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/ergonyc/mambaforge/envs/nb-cuda/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/ergonyc/mambaforge/envs/nb-cuda/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training:   0%|          | 0/300 [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


Exception raised during training. <class 'NameError'> 1


SystemExit: 1

/home/ergonyc/mambaforge/envs/nb-cuda/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
scanvi_model_filename = local_data_path / f"asap-{dataset_team}.SN.05_scanvi_model"

scanvi_model.save(scanvi_model_filename, overwrite=True)
# 6. Save the latent space
# output file neame
sn_scanvi_filename = local_data_path / f"asap-{dataset_team}.SN.05_scanvi.h5ad"

adata.write_h5ad(filename=args.adata_output, compression="gzip")
# 7. Save the cell types to feather
# adata.obs[[args.predictions_key]].to_feather(args.output_cell_types_file, compression="gzip")
# 7. Save the cell types to parquet

output_cell_types_file = (
    local_data_path / f"asap-{dataset_team}.SN.05_scanvi_cell_types.parquet"
)

adata.obs[[args.predictions_key]].to_parquet(output_cell_types_file, compression="gzip")

: 

In [ ]:
adata.obs.columns

Index(['background_fraction', 'cell_probability', 'cell_size',
       'droplet_efficiency', 'n_genes_by_counts', 'total_counts',
       'total_counts_rb', 'pct_counts_rb', 'total_counts_mt', 'pct_counts_mt',
       'doublet_score', 'sample', 'batch', 'team', 'dataset', 'batch_id',
       'S_score', 'G2M_score', 'phase', 'brain_region', 'brain_region_simple',
       'case_id', 'condition_id', 'region_level_1', 'region_level_2',
       '_cell_type', '_phenotype', '_rho', '_prob', '_class_name',
       '_subclass_name', '_supertype_name', '__scvi_batch', '__scvi_labels',
       '_C_scANVI', '_leiden_res_0.05', '_leiden_res_0.10', '_leiden_res_0.20',
       '_leiden_res_0.40', '_scvi_batch', '_scvi_labels', 'atlas_identifier',
       'leiden_2', 'leiden', 'leiden_05', 'cell_type', 'phenotype', 'rho',
       'prob', 'class_name', 'subclass_name', 'supertype_name'],
      dtype='object')

In [ ]:
adata.obsm.columns

AttributeError: 'AxisArrays' object has no attribute 'columns'

In [ ]:
adata.obsm

AxisArrays with keys: X_pca, X_scvi, X_umap, _X_pca, _X_pca_harmony, _X_scANVI, _X_scVI, _X_umap, gene_expression_encoding

In [ ]:
####################################

import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import sys, subprocess, importlib, warnings, math, os

from pathlib import Path

import torch
import scvi

In [ ]:
##
# pip3 install -U scvi-tools[cuda]  # gets jax and jaxlib, updates cuda
# pip3 install -U scib-metrics
#
#
####################################

###########
# ## STEP 0. Workspace Setup

# set general folder paths
HOME = Path.home()
WS_ROOT = HOME / "workspace"
DATA_DIR = WS_ROOT / "data"
WS_FILES = WS_ROOT / "ws_files"

In [ ]:
## Build and set path to desired dataset
DATASETS_PATH = WS_ROOT / "01_PMDBS" / "PMDBS_sc_rnaseq"

workflow = "pmdbs_sc_rnaseq"
dataset_team = "cohort"
dataset_source = "pmdbs"
dataset_type = "sc-rnaseq"

bucket_name = f"asap-curated-{dataset_team}-{dataset_source}-{dataset_type}"
dataset_name = f"asap-{dataset_team}-{dataset_source}-{dataset_type}"

dataset_path = DATASETS_PATH / bucket_name / workflow
print("Dataset Path:", dataset_path)

# Build the folder path to the cohort analysis directory
cohort_analysis_path = dataset_path / "cohort_analysis"

# Preview the directory contents
# Define a local path for workshop files
local_data_path = WS_FILES / "case_study_01"

# map my cells directories
mapmycells_input_dir = local_data_path / "mapmycells/input"
mapmycells_output_dir = local_data_path / "mapmycells/output"

# other directories
resources_path = local_data_path / "resources"
plots_path = local_data_path / "output_plots"
output_path = local_data_path / "output_matrices"

# Make sure the directories exists
os.makedirs(mapmycells_input_dir, exist_ok=True)
os.makedirs(mapmycells_output_dir, exist_ok=True)
os.makedirs(resources_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)
os.makedirs(plots_path, exist_ok=True)

# Create the directory if it doesn't already exist
if not local_data_path.exists():
    local_data_path.mkdir(parents=True)

print(f"Local data directory ready at: {local_data_path}")

Dataset Path: /home/ergonyc/workspace/01_PMDBS/PMDBS_sc_rnaseq/asap-curated-cohort-pmdbs-sc-rnaseq/pmdbs_sc_rnaseq
Local data directory ready at: /home/ergonyc/workspace/ws_files/case_study_01


In [ ]:
###########

In [ ]:
sn_full_raw_filename = local_data_path / f"asap-{dataset_team}.SN.01_full_raw.h5ad"

In [ ]:
sn_full_norm_filename = local_data_path / f"asap-{dataset_team}.SN.02_raw_norm.h5ad"

In [ ]:
#################################
sn_processed_filename = local_data_path / f"asap-{dataset_team}.SN.02_processed.h5ad"

# Save the anndata object
sn_integrated_filename = local_data_path / f"asap-{dataset_team}.SN.03_scvi.h5ad"

# output file neame
sn_mmc_pheno_filename = (
    local_data_path / f"asap-{dataset_team}.SN.04_mmc_processed.h5ad"
)


# have to use ENSG ids for this mapmycells taxonomy
# prep data

In [ ]:
###########
adata = sc.read_h5ad(sn_mmc_pheno_filename)
# ## STEP 3. make SN integrate with scVI (.SN.03_scvi.h5ad)
batch_key = "sample"
n_layers = 2
n_comps = adata.obsm["X_pca"].shape[1]
n_latent = n_comps  # defined above
predictions_key = "C_scANVI"
###
print(torch.cuda.is_available())

scvi.settings.seed = 0
torch.set_float32_matmul_precision("high")


scvi_model_filename = local_data_path / f"asap-{dataset_team}.SN.03_scvi_model"
vae = scvi.model.SCVI.load(scvi_model_filename)

Seed set to 0


True
INFO     File /home/ergonyc/workspace/ws_files/case_study_01/asap-cohort.SN.03_scvi_model/model.pt already         
         downloaded                                                                                                


/home/ergonyc/mambaforge/envs/nb-cuda/lib/python3.13/site-packages/scvi/model/base/_base_model.py:869: UserWarning: Save path contains no saved anndata and no adata was passed. Model will be loaded without anndata.
  ) = _load_saved_files(


In [ ]:
def label_with_scanvi(
    adata: sc.AnnData,
    model: scvi.model.SCVI,
    num_workers: int,
    latent_key: str | None = None,
    predictions_key: str | None = None,
    workflow_name: str = "generic_000",
) -> tuple[sc.AnnData, scvi.model.SCANVI]:
    """
    Fit scANVI model to AnnData object
    """

    # Fixed parameters
    scanvi_epochs = 300
    batch_size = 1024
    accelerator = "gpu"
    dispersion = "gene-cell"  # "gene"
    gene_likelihood = "zinb"
    latent_distribution = "normal"
    early_stopping = True
    early_stopping_patience = 20

    if latent_key is None:
        latent_key = f"X_scANVI"
    if predictions_key is None:
        predictions_key = f"C_scANVI"

    # if adata.n_obs > threshold_cells:
    #     plan_kwargs = {"lr": 1e-4}
    #     gradient_clip_val = 5.0
    #     print(f"AnnData object contains {adata.n_obs} which is > {threshold_cells}")
    #     print(f"--- Using learning rate: {plan_kwargs}")
    #     print(f"--- Using gradient clipping: {gradient_clip_val}")
    # else:
    # Defaults
    plan_kwargs = {"lr": 1e-3}
    gradient_clip_val = None
    # print(f"AnnData object contains {adata.n_obs} which is < {threshold_cells}")
    # print(f"--- Using default learning rate: {plan_kwargs}")
    # print(f"--- Using default gradient clipping: {gradient_clip_val}")

    print("Generating scANVI model from scVI")
    scanvi_model = scvi.model.SCANVI.from_scvi_model(
        model,
        adata=adata,
        labels_key="cell_type",
        unlabeled_category="Unknown",
    )

    print("Training scANVI model")
    scanvi_model.train(
        accelerator=accelerator,
        max_epochs=scanvi_epochs,
        early_stopping=early_stopping,
        early_stopping_patience=early_stopping_patience,
        datasplitter_kwargs={"num_workers": num_workers},
        gradient_clip_val=gradient_clip_val,
        plan_kwargs=plan_kwargs,
    )

    print("Generating scANVI latents and predictions")
    adata.obsm[latent_key] = scanvi_model.get_latent_representation(adata)
    adata.obs[predictions_key] = scanvi_model.predict(adata)

    return (adata, scanvi_model)

In [ ]:
num_workers = 0  # Pytorch bug unable to mmap solution https://github.com/pytorch/pytorch/issues/92134
scvi.settings.dl_num_workers = num_workers
print(f"Using {scvi.settings.dl_num_workers} workers")

# 4. Get scANVI model
workflow_name = "case_study_01"
adata, scanvi_model = label_with_scanvi(
    adata,
    vae,
    num_workers,
    workflow_name=workflow_name,
    predictions_key=predictions_key,
)

# 5. Save the integrated adata and scANVI model

Using 0 workers
Generating scANVI model from scVI
INFO     Model was loaded without AnnData. Setting up provided AnnData using saved registry.                       
Training scANVI model
INFO     Training for 300 epochs.                                                                                  


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/ergonyc/mambaforge/envs/nb-cuda/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/ergonyc/mambaforge/envs/nb-cuda/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training:   0%|          | 0/300 [00:00<?, ?it/s]

Monitored metric elbo_validation did not improve in the last 20 records. Best score: 141.499. Signaling Trainer to stop.
Generating scANVI latents and predictions


In [ ]:
scanvi_model_filename = local_data_path / f"asap-{dataset_team}.SN.05_scanvi_model"

scanvi_model.save(scanvi_model_filename, overwrite=True)
# 6. Save the latent space
# output file neame
sn_scanvi_filename = local_data_path / f"asap-{dataset_team}.SN.05_scanvi.h5ad"

adata.write_h5ad(filename=sn_scanvi_filename, compression="gzip")
# 7. Save the cell types to feather
# adata.obs[[args.predictions_key]].to_feather(args.output_cell_types_file, compression="gzip")
# 7. Save the cell types to parquet

output_cell_types_file = (
    local_data_path / f"asap-{dataset_team}.SN.05_scanvi_cell_types.parquet"
)

adata.obs[[predictions_key]].to_parquet(output_cell_types_file, compression="gzip")

In [ ]:
adata

AnnData object with n_obs × n_vars = 258505 × 2764
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'n_genes_by_counts', 'total_counts', 'total_counts_rb', 'pct_counts_rb', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'sample', 'batch', 'team', 'dataset', 'batch_id', 'S_score', 'G2M_score', 'phase', 'brain_region', 'brain_region_simple', 'case_id', 'condition_id', 'region_level_1', 'region_level_2', '_cell_type', '_phenotype', '_rho', '_prob', '_class_name', '_subclass_name', '_supertype_name', '__scvi_batch', '__scvi_labels', '_C_scANVI', '_leiden_res_0.05', '_leiden_res_0.10', '_leiden_res_0.20', '_leiden_res_0.40', '_scvi_batch', '_scvi_labels', 'atlas_identifier', 'leiden_2', 'leiden', 'leiden_05', 'cell_type', 'phenotype', 'rho', 'prob', 'class_name', 'subclass_name', 'supertype_name', 'C_scANVI'
    var: 'feature_type', 'genome', 'gene_id', 'mt', 'rb', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'n_cells'
    uns: '